In [1]:
# Load packages

import pandas as pd
import numpy as np

from pathlib import Path
import json
import joblib
import warnings
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")
np.random.seed(42)

In [2]:
# Core Helper Functions for Neural Network Notebook

def ks_statistic(y_true, y_score):
    temp = pd.DataFrame({
        "y_true": y_true,
        "y_score": y_score
    }).sort_values("y_score", ascending=False)

    temp["good"] = (temp["y_true"] == 0).astype(int)
    temp["bad"] = (temp["y_true"] == 1).astype(int)

    temp["cum_good"] = temp["good"].cumsum() / temp["good"].sum()
    temp["cum_bad"] = temp["bad"].cumsum() / temp["bad"].sum()

    return (temp["cum_bad"] - temp["cum_good"]).abs().max()


def calculate_classification_diagnostics(
    y_train,
    p_train,
    y_val,
    p_val,
    threshold=0.50
):
    rows = []

    for dataset_name, y_true, p_score in [
        ("train", y_train, p_train),
        ("validation", y_val, p_val)
    ]:

        y_pred = (p_score >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "dataset": dataset_name,
            "threshold": threshold,
            "auc": roc_auc_score(y_true, p_score),
            "gini": 2 * roc_auc_score(y_true, p_score) - 1,
            "ks": ks_statistic(y_true, p_score),
            "pr_auc": average_precision_score(y_true, p_score),
            "log_loss": log_loss(y_true, p_score),
            "brier_score": brier_score_loss(y_true, p_score),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp
        })

    return pd.DataFrame(rows)


def get_next_model_number(model_registry, prefix):
    existing_numbers = []

    for model_num in model_registry.keys():
        if model_num.startswith(prefix):
            number_part = model_num.replace(prefix, "")
            if number_part.isdigit():
                existing_numbers.append(int(number_part))

    return max(existing_numbers) + 1 if existing_numbers else 1


def save_model_artifact(model_registry, model_id, model_dir, config_dir):
    record = model_registry[model_id]

    model_path = model_dir / f"{model_id}.joblib"
    config_path = config_dir / f"{model_id}_config.json"

    joblib.dump(record["model"], model_path)

    metadata = record.get("metadata", {}).copy()

    config = {
        "model_id": model_id,
        "model_class": str(type(record["model"]).__name__),
        "metadata": metadata,
        "features": metadata.get("features", None),
        "num_features": metadata.get("num_features", None),
    }

    if "diagnostics" in record:
        try:
            config["diagnostics_preview"] = record["diagnostics"].to_dict(orient="records")
        except Exception:
            pass

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4, default=str)


def run_nn_model(
    model_registry,
    model,
    model_number,
    model_name,
    X_train,
    X_val,
    y_train,
    y_val,
    features,
    threshold=0.50,
    analyst_comments="",
    run_permutation=True,
    permutation_scoring="roc_auc",
    permutation_repeats=5,
    display_outputs=True
):

    if model_number in model_registry:
        raise ValueError(f"{model_number} already exists.")

    features = list(features)

    model.fit(X_train[features], y_train)

    p_train = model.predict_proba(X_train[features])[:, 1]
    p_val = model.predict_proba(X_val[features])[:, 1]

    diagnostics_table = calculate_classification_diagnostics(
        y_train=y_train,
        p_train=p_train,
        y_val=y_val,
        p_val=p_val,
        threshold=threshold
    )

    if run_permutation:
        perm_result = permutation_importance(
            model,
            X_val[features],
            y_val,
            scoring=permutation_scoring,
            n_repeats=permutation_repeats,
            random_state=42,
            n_jobs=-1
        )

        permutation_importance_df = pd.DataFrame({
            "variable": features,
            "permutation_importance_mean": perm_result.importances_mean,
            "permutation_importance_std": perm_result.importances_std
        }).sort_values(
            "permutation_importance_mean",
            ascending=False
        ).reset_index(drop=True)
    else:
        permutation_importance_df = pd.DataFrame()

    metadata = {
        "model_number": model_number,
        "model_name": model_name,
        "model_class": type(model).__name__,
        "features": features,
        "num_features": len(features),
        "threshold": threshold,
        "parameters": model.get_params(),
        "n_iter": getattr(model, "n_iter_", None),
        "loss": getattr(model, "loss_", None),
        "analyst_comments": analyst_comments
    }

    model_registry[model_number] = {
        "metadata": metadata,
        "model": model,
        "diagnostics": diagnostics_table,
        "feature_importance": pd.DataFrame(),
        "permutation_importance": permutation_importance_df,
        "p_train": p_train,
        "p_val": p_val
    }

    if display_outputs:
        print("=" * 100)
        print(f"{model_number}: {model_name}")
        print("=" * 100)

        print("\nDIAGNOSTICS:")
        display(diagnostics_table)

        print("\nITERATIONS:")
        print(getattr(model, "n_iter_", None))

        print("\nPERMUTATION IMPORTANCE:")
        display(permutation_importance_df.head(15))

        if analyst_comments:
            print("\nANALYST COMMENTS:")
            print(analyst_comments)

    return model_registry

In [3]:
# Paths

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "saved_models"
CONFIG_DIR = OUTPUT_DIR / "model_configs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)
print("Output Directory:", OUTPUT_DIR)

Project Root: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab
Data Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/data
Output Directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs


In [4]:
# Load dataset

df_scaled = pd.read_parquet(DATA_DIR / "df_scaled_model_ready.parquet")

print("Scaled dataset shape:", df_scaled.shape)
display(df_scaled.head())

Scaled dataset shape: (149390, 16)


,SeriousDlqin2yrs,age,age_sq,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTimes90DaysLate,NumberOfOpenCreditLinesAndLoans,NumberRealEstateLoansOrLines,NumberOfDependents_median,NumberOfDependents_missing_flag,MonthlyIncome_median,MonthlyIncome_missing_flag,DebtRatio_log,DebtRatio_high_flag,RevolvingUtilization_log,RevolvingUtilization_high_flag
0,1,-0.496191,-0.578484,0.416854,-0.055768,-0.062235,0.879798,4.404219,1.136564,-0.162167,0.208608,-0.493119,-0.131442,-0.489465,1.283222,-0.049896
1,0,-0.835742,-0.843467,-0.102229,-0.055768,-0.062235,-0.872364,-0.904611,0.234254,-0.162167,-0.296205,-0.493119,-0.679415,-0.489465,1.692974,-0.049896
2,0,-0.971562,-0.940732,0.157313,-0.055768,0.199123,-1.261734,-0.904611,-0.668056,-0.162167,-0.261983,-0.493119,-0.717896,-0.489465,1.031595,-0.049896
3,0,-1.514844,-1.279910,-0.102229,-0.055768,-0.062235,-0.677680,-0.904611,-0.668056,-0.162167,-0.242008,-0.493119,-0.771337,-0.489465,-0.147833,-0.049896
4,0,-0.224551,-0.344051,0.157313,-0.055768,-0.062235,-0.288310,-0.019806,-0.668056,-0.162167,4.425811,-0.493119,-0.783805,-0.489465,1.589907,-0.049896


In [5]:
# Define Target + Candidate Features

target = "SeriousDlqin2yrs"

candidate_features = [
    col for col in df_scaled.columns
    if col != target
]

print("Target:", target)
print("Number of candidate features:", len(candidate_features))
print(candidate_features)

Target: SeriousDlqin2yrs
Number of candidate features: 15
['age', 'age_sq', 'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfTimes90DaysLate', 'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines', 'NumberOfDependents_median', 'NumberOfDependents_missing_flag', 'MonthlyIncome_median', 'MonthlyIncome_missing_flag', 'DebtRatio_log', 'DebtRatio_high_flag', 'RevolvingUtilization_log', 'RevolvingUtilization_high_flag']


In [6]:
# Train / Validation Split

X = df_scaled[candidate_features].copy()
y = df_scaled[target].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("Train bad rate:", y_train.mean())
print("Val bad rate  :", y_val.mean())

X_train: (104573, 15)
X_val  : (44817, 15)
Train bad rate: 0.06699626098514913
Val bad rate  : 0.06700582368297744


In [7]:
# Initialize registry and model

nn_registry = {}



## Neural Network Modeling Approach: Rationale, Assumptions, and Design Choices

Neural networks were introduced in this project as a nonlinear benchmark model family to complement the earlier Logistic Regression, tree-based ensemble, and Naive Bayes models. The objective was not simply to maximize validation AUC, but to evaluate whether a flexible representation-learning model could extract additional predictive signal from the structured credit-risk dataset.

Unlike Logistic Regression, which assumes a linear relationship between predictors and the log-odds of default, neural networks can learn nonlinear interactions and threshold effects automatically through hidden layers. This is particularly relevant in credit-risk data, where relationships such as utilization stress, delinquency frequency, debt burden, and age effects may not behave linearly across the full population.

Unlike tree-based models, neural networks do not rely on recursive partitioning rules. Instead, they learn weighted transformations of the feature space through multiple layers of neurons. This makes them a useful independent benchmark because strong performance would indicate that predictive structure exists beyond linear models and tree split logic.

---

## Why Scaled Inputs

The neural network models were trained on the scaled model-ready dataset rather than raw or binned data.

This decision was made because gradient-based optimization methods (such as Adam) perform best when inputs are numerically stable and approximately standardized. Large differences in variable magnitude can slow convergence or create unstable weight updates.

Scaled continuous features also preserve more numeric detail than aggressively binned variables, allowing the network to learn smoother nonlinear relationships rather than only discrete bucket effects.

---

## Why a Simple Architecture

The first benchmark model uses:

- One hidden layer
- 32 neurons
- ReLU activation
- Adam optimizer
- L2 regularization through `alpha`
- Early stopping

This intentionally modest design was chosen for several reasons:

### 1. Structured Tabular Data Usually Does Not Need Deep Networks

For many business tabular datasets, especially medium-sized structured credit data, extremely deep networks often do not outperform simpler architectures or boosting models.

A shallow network frequently captures most usable nonlinear signal without unnecessary complexity.

### 2. Overfitting Risk

Credit-risk datasets often contain strong dominant predictors. Deep networks can memorize noise rather than learn stable patterns. Starting small creates a cleaner baseline before testing more complex designs.

### 3. Interpretability and Governance

Although neural networks are less interpretable than logistic models, a smaller architecture is easier to justify, replicate, and govern than a deep black-box network.

---

## Why ReLU Activation 
ReLU (Rectified Linear Unit) is the modern default hidden-layer activation because:

- trains faster than sigmoid/tanh
- reduces vanishing gradient problems
- computationally efficient
- works well on scaled numeric tabular data

---

## Why Probabilistic Output

The target variable is binary, so the neural network needs to output an estimated probability of serious delinquency. In `sklearn.MLPClassifier`, this is handled through the classifier’s probabilistic output via `predict_proba`.

This allows direct comparison with earlier models using AUC, KS, PR-AUC, Log Loss, and Brier Score.

---

## Why Log-Loss Style Optimization

For binary classification, `MLPClassifier` optimizes a log-loss style objective. This is appropriate because the model is learning probability estimates, not just hard class labels. Log-loss penalizes confident wrong predictions more heavily, which is important for credit-risk probability modeling.

---

## Why Adam Optimizer

Adam was selected because it is robust, fast, and widely used in practical machine learning settings. It adaptively adjusts learning rates during training and generally works well without extensive manual tuning.

This makes it an appropriate first optimizer for benchmarking.

---

## Why Early Stopping

Early stopping monitors validation performance during training and stops when improvement stalls.

This was used to reduce overfitting, shorten training time, and keep the model closer to its strongest validation state. This is especially important in tabular projects where too many iterations can degrade out-of-sample performance.
---

## What We Expect Relative to Other Models

Based on industry experience with structured tabular credit data:

- Neural networks may outperform simpler Naive Bayes models.
- They may challenge logistic regression if nonlinear effects are strong.
- They often struggle to consistently beat top gradient boosting models such as LightGBM or CatBoost on tabular datasets.

That expectation is precisely why benchmarking is necessary rather than assumed.

---

## Business Interpretation

If neural networks materially outperform classical models, it suggests hidden nonlinear relationships and interactions exist in the borrower population.

If they do not outperform boosting or logistic models, that is also valuable. It would imply that simpler or more governable methods already capture most of the usable signal.

Both outcomes are informative.

---

## Project Philosophy

Neural networks are included here not as hype, but as a disciplined benchmark within a broader model risk framework. Every model family is tested under comparable data splits and evaluation metrics so that final model selection is evidence-based rather than trend-based.

In [8]:
# NN001 - Baseline Neural Network

nn_model = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=42
)

nn_registry = run_nn_model(
    model_registry=nn_registry,
    model=nn_model,
    model_number="NN001",
    model_name="Baseline Neural Network - MLP (32)",
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    y_val=y_val,
    features=candidate_features,
    analyst_comments=(
        "Baseline sklearn MLPClassifier with one hidden layer of 32 neurons, "
        "ReLU activation, Adam optimizer, L2 regularization, and early stopping."
    ),
    run_permutation=True,
    permutation_repeats=5,
    display_outputs=True
)

NN001: Baseline Neural Network - MLP (32)

DIAGNOSTICS:


,dataset,threshold,auc,gini,ks,pr_auc,log_loss,brier_score,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive
0,train,0.5,0.858081,0.716161,0.556731,0.382491,0.181097,0.049757,0.936858,0.582752,0.202541,0.300604,96551,1016,5587,1419
1,validation,0.5,0.860711,0.721421,0.565110,0.397369,0.179720,0.049412,0.937234,0.594433,0.199134,0.298329,41406,408,2405,598



ITERATIONS:
20

PERMUTATION IMPORTANCE:


,variable,permutation_importance_mean,permutation_importance_std
0,RevolvingUtilization_log,0.091649,0.002300
1,NumberOfTime30-59DaysPastDueNotWorse,0.030855,0.001187
2,NumberOfTimes90DaysLate,0.026689,0.000899
3,DebtRatio_log,0.022875,0.001099
4,DebtRatio_high_flag,0.017186,0.001417
5,NumberOfTime60-89DaysPastDueNotWorse,0.013230,0.000816
6,age,0.011126,0.000647
7,NumberOfOpenCreditLinesAndLoans,0.006246,0.000632
8,MonthlyIncome_missing_flag,0.005767,0.000841
9,NumberRealEstateLoansOrLines,0.003431,0.000348



ANALYST COMMENTS:
Baseline sklearn MLPClassifier with one hidden layer of 32 neurons, ReLU activation, Adam optimizer, L2 regularization, and early stopping.


In [9]:
# Neural Network Model Grid

nn_grid = [
    {
        "model_number": "NN002",
        "model_name": "MLP - Medium Network (64, 32)",
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.0001,
        "learning_rate_init": 0.001,
        "features": candidate_features,
        "comments": "Tests whether a deeper/wider network improves nonlinear learning."
    },
    {
        "model_number": "NN003",
        "model_name": "MLP - Regularized Network (32, 16)",
        "hidden_layer_sizes": (32, 16),
        "alpha": 0.001,
        "learning_rate_init": 0.001,
        "features": candidate_features,
        "comments": "Tests stronger L2 regularization with a moderately deep architecture."
    },
    {
        "model_number": "NN004",
        "model_name": "MLP - Compact Top Features",
        "hidden_layer_sizes": (32,),
        "alpha": 0.0001,
        "learning_rate_init": 0.001,
        "features": [
            "RevolvingUtilization_log",
            "NumberOfTime30-59DaysPastDueNotWorse",
            "NumberOfTimes90DaysLate",
            "DebtRatio_log",
            "DebtRatio_high_flag",
            "NumberOfTime60-89DaysPastDueNotWorse",
            "age",
            "NumberOfOpenCreditLinesAndLoans"
        ],
        "comments": "Tests whether a compact high-signal feature set improves stability."
    },
    {
        "model_number": "NN005",
        "model_name": "MLP - Smaller Network Strong Regularization",
        "hidden_layer_sizes": (16,),
        "alpha": 0.001,
        "learning_rate_init": 0.001,
        "features": candidate_features,
        "comments": "Tests whether a simpler regularized network generalizes better."
    },
    {
        "model_number": "NN006",
        "model_name": "MLP - Lower Learning Rate",
        "hidden_layer_sizes": (32,),
        "alpha": 0.0001,
        "learning_rate_init": 0.0005,
        "features": candidate_features,
        "comments": "Tests whether slower optimization improves validation performance."
    },
]

In [10]:
# Run Neural Network Grid

for params in nn_grid:

    nn_model = MLPClassifier(
        hidden_layer_sizes=params["hidden_layer_sizes"],
        activation="relu",
        solver="adam",
        alpha=params["alpha"],
        batch_size=256,
        learning_rate_init=params["learning_rate_init"],
        max_iter=150,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=10,
        random_state=42
    )

    nn_registry = run_nn_model(
        model_registry=nn_registry,
        model=nn_model,
        model_number=params["model_number"],
        model_name=params["model_name"],
        X_train=X_train,
        X_val=X_val,
        y_train=y_train,
        y_val=y_val,
        features=params["features"],
        analyst_comments=params["comments"],
        run_permutation=True,
        permutation_repeats=5,
        display_outputs=False
    )

print("Neural network grid complete.")

Neural network grid complete.


In [11]:
# Neural Network Summary Table

nn_summary_rows = []

for model_num, model_data in nn_registry.items():

    meta = model_data["metadata"]
    diag = model_data["diagnostics"]
    params = meta.get("parameters", {})

    train_row = diag.loc[diag["dataset"] == "train"].iloc[0]
    val_row = diag.loc[diag["dataset"] == "validation"].iloc[0]

    nn_summary_rows.append({
        "model_number": model_num,
        "model_name": meta["model_name"],
        "hidden_layer_sizes": params.get("hidden_layer_sizes"),
        "alpha": params.get("alpha"),
        "learning_rate_init": params.get("learning_rate_init"),
        "num_features": meta.get("num_features"),
        "n_iter": meta.get("n_iter"),
        "loss": meta.get("loss"),

        "train_auc": train_row["auc"],
        "val_auc": val_row["auc"],
        "auc_gap": train_row["auc"] - val_row["auc"],

        "train_ks": train_row["ks"],
        "val_ks": val_row["ks"],
        "ks_gap": train_row["ks"] - val_row["ks"],

        "val_pr_auc": val_row["pr_auc"],
        "val_brier": val_row["brier_score"],
        "val_log_loss": val_row["log_loss"],
        "val_accuracy": val_row["accuracy"],
        "val_precision": val_row["precision"],
        "val_recall": val_row["recall"],
        "val_f1": val_row["f1"]
    })

nn_summary_df = (
    pd.DataFrame(nn_summary_rows)
    .sort_values(["val_auc", "val_ks", "val_pr_auc"], ascending=False)
    .reset_index(drop=True)
)

display(nn_summary_df)

,model_number,model_name,hidden_layer_sizes,alpha,learning_rate_init,num_features,n_iter,loss,train_auc,val_auc,...,train_ks,val_ks,ks_gap,val_pr_auc,val_brier,val_log_loss,val_accuracy,val_precision,val_recall,val_f1
0,NN003,"MLP - Regularized Network (32, 16)","(32, 16)",0.0010,0.0010,15,21,0.177897,0.865068,0.863171,...,0.571154,0.571177,-0.000022,0.394046,0.049317,0.178940,0.937278,0.598969,0.193473,0.292474
1,NN002,"MLP - Medium Network (64, 32)","(64, 32)",0.0001,0.0010,15,16,0.177289,0.861857,0.862989,...,0.564054,0.574544,-0.010490,0.401562,0.049170,0.178788,0.937457,0.596899,0.205128,0.305328
2,NN005,MLP - Smaller Network Strong Regularization,"(16,)",0.0010,0.0010,15,26,0.179148,0.858679,0.861678,...,0.555475,0.569038,-0.013563,0.384986,0.049758,0.180141,0.936096,0.564055,0.203796,0.299413
3,NN001,Baseline Neural Network - MLP (32),"(32,)",0.0001,0.0010,15,20,0.178090,0.858081,0.860711,...,0.556731,0.565110,-0.008379,0.397369,0.049412,0.179720,0.937234,0.594433,0.199134,0.298329
4,NN004,MLP - Compact Top Features,"(32,)",0.0001,0.0010,8,15,0.180025,0.847657,0.851810,...,0.537181,0.550170,-0.012989,0.377170,0.050333,0.183981,0.936230,0.597315,0.148185,0.237460
5,NN006,MLP - Lower Learning Rate,"(32,)",0.0001,0.0005,15,19,0.179634,0.846853,0.849719,...,0.532555,0.544310,-0.011755,0.376434,0.050500,0.184745,0.936497,0.603974,0.151848,0.242682


In [12]:
# Save Neural Network Champion Artifact

save_model_artifact(
    model_registry=nn_registry,
    model_id="NN003",
    model_dir=MODEL_DIR,
    config_dir=CONFIG_DIR
)

In [13]:
# Export Neural Network Champion Summary

nn_champion_id = "NN003"

nn_champion_path = OUTPUT_DIR / "04d_neural_network_champion_summary.xlsx"

with pd.ExcelWriter(nn_champion_path, engine="openpyxl") as writer:

    nn_summary_df.to_excel(
        writer,
        sheet_name="NN_Model_Comparison",
        index=False
    )

    nn_registry[nn_champion_id]["diagnostics"].to_excel(
        writer,
        sheet_name="NN003_Diagnostics",
        index=False
    )

    nn_registry[nn_champion_id]["permutation_importance"].to_excel(
        writer,
        sheet_name="NN003_Permutation",
        index=False
    )

print("Saved:", nn_champion_path)

Saved: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/04d_neural_network_champion_summary.xlsx


## Neural Network Summary

Neural networks were tested as a nonlinear benchmark model family using the scaled model-ready dataset. The goal was to evaluate whether a feed-forward multilayer perceptron could capture additional nonlinear borrower-risk patterns beyond logistic regression, Naive Bayes, and tree-based models.

Several architectures were tested, including a shallow baseline network, a larger two-layer network, a more regularized two-layer network, a compact top-feature network, and a lower learning-rate variant. The strongest model was **NN003**, a regularized `(32, 16)` multilayer perceptron.

NN003 achieved the best validation AUC among the neural-network candidates, showing that moderate depth plus stronger L2 regularization provided the best balance of flexibility and generalization. Larger or slower-learning models did not materially improve results, and the compact top-feature model underperformed, indicating that the neural network benefited from the broader scaled feature set.

The neural network performed strongly relative to Naive Bayes and was competitive with other classical models, but it did not outperform the best boosting models. This is consistent with expectations for structured tabular credit-risk data, where gradient-boosted trees often remain difficult to beat.

**Final neural network champion:** NN003  
**Key takeaway:** controlled network complexity and regularization mattered more than simply increasing model size.